# 01 - Data Preparation & Chunked ETL Pipeline (Google Colab)

### Customer Support on Twitter Dataset Preprocessing
**Project:** Customer Support RAG Chatbot  
**Execution Environment:** Google Colab (Free CPU / T4 GPU)  
**Storage Backend:** Google Drive (`/content/drive/MyDrive/chatbot_data/`)

---
### Objectives
1. Mount Google Drive for persistent storage of raw, cleaned, and tokenized datasets.
2. Download the Kaggle **Customer Support on Twitter** (`thoughtvector/customer-support-on-twitter`) dataset.
3. Stream the ~1GB `twcs.csv` in chunks (`chunksize=50,000`) to completely eliminate Out-Of-Memory (OOM) crashes on Colab's 12GB RAM limit.
4. Filter high-value customer inquiries and verified brand responses (@AppleSupport, @AmazonHelp, @Uber_Support, @SpotifyCares, @Delta).
5. Clean, anonymize, and pair inquiries with brand solutions into standard JSONL format.
6. Export `cleaned_customer_support_sample.jsonl` (50,000 high-quality QA pairs) to Google Drive.



In [1]:
# Step 1: Mount Google Drive
from google.colab import drive
import os
import sys

drive.mount('/content/drive')

DRIVE_BASE_DIR = "/content/drive/MyDrive/chatbot_data"
os.makedirs(f"{DRIVE_BASE_DIR}/raw", exist_ok=True)
os.makedirs(f"{DRIVE_BASE_DIR}/processed", exist_ok=True)
os.makedirs(f"{DRIVE_BASE_DIR}/tokenized", exist_ok=True)

print(f"[OK] Google Drive mounted successfully.")
print(f"[OK] Workspace created at: {DRIVE_BASE_DIR}")



Mounted at /content/drive
[OK] Google Drive mounted successfully.
[OK] Workspace created at: /content/drive/MyDrive/chatbot_data


In [2]:
# Step 2: Install required lightweight utilities
!pip install -q kaggle pandas tqdm



### Step 3: Kaggle Authentication & Dataset Download
Provide your `kaggle.json` credentials either by placing them in `/content/drive/MyDrive/kaggle.json` or by setting Colab Secrets (`KAGGLE_USERNAME`, `KAGGLE_KEY`).



In [3]:
import os
import json
import shutil
from pathlib import Path

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)
kaggle_json = kaggle_dir / "kaggle.json"
kaggle_token = kaggle_dir / "access_token"

# 1. Support new Kaggle API Token (KGAT_...) from env, Colab Secrets, or Drive
drive_token = Path("/content/drive/MyDrive/kaggle_access_token")
if "KAGGLE_API_TOKEN" in os.environ:
    kaggle_token.write_text(os.environ["KAGGLE_API_TOKEN"].strip())
    kaggle_token.chmod(0o600)
    print("[OK] Configured Kaggle API Token from environment variable.")
elif drive_token.exists():
    shutil.copy(drive_token, kaggle_token)
    kaggle_token.chmod(0o600)
    print("[OK] Loaded Kaggle API Token from Google Drive.")
else:
    try:
        from google.colab import userdata
        kgat = userdata.get("KAGGLE_API_TOKEN")
        if kgat:
            os.environ["KAGGLE_API_TOKEN"] = kgat.strip()
            kaggle_token.write_text(kgat.strip())
            kaggle_token.chmod(0o600)
            print("[OK] Configured Kaggle API Token from Colab Secrets.")
    except Exception:
        pass

# 2. Fallback to legacy kaggle.json
drive_kaggle = Path("/content/drive/MyDrive/kaggle.json")
if not kaggle_token.exists():
    if drive_kaggle.exists():
        shutil.copy(drive_kaggle, kaggle_json)
        kaggle_json.chmod(0o600)
        print("[OK] Loaded kaggle.json from Google Drive.")
    else:
        try:
            from google.colab import userdata
            k_user = userdata.get("KAGGLE_USERNAME")
            k_key = userdata.get("KAGGLE_KEY")
            if k_user and k_key:
                with open(kaggle_json, "w") as f:
                    json.dump({"username": k_user, "key": k_key}, f)
                kaggle_json.chmod(0o600)
                print("[OK] Configured kaggle credentials from Colab Secrets.")
        except Exception:
            pass

if not kaggle_token.exists() and not kaggle_json.exists():
    print("[!] No automatic Kaggle credentials found.")
    print("[!] Please set KAGGLE_API_TOKEN in Colab Secrets or provide kaggle.json.")

# Download dataset if not already present
raw_csv_path = Path("/content/twcs.csv")
drive_csv_path = Path(f"{DRIVE_BASE_DIR}/raw/twcs.csv")

if drive_csv_path.exists():
    print(f"[OK] Found twcs.csv in Drive: {drive_csv_path}")
    raw_csv_path = drive_csv_path
elif raw_csv_path.exists():
    print(f"[OK] Found local twcs.csv in /content")
else:
    print("[*] Downloading dataset from Kaggle...")
    !kaggle datasets download -d thoughtvector/customer-support-on-twitter -p /content --unzip
    if raw_csv_path.exists():
        print("[*] Backing up raw twcs.csv to Google Drive for future sessions...")
        shutil.copy(raw_csv_path, drive_csv_path)
        print("[OK] Backup complete.")


[OK] Configured Kaggle API Token from Colab Secrets.
[*] Downloading dataset from Kaggle...
Dataset URL: https://www.kaggle.com/datasets/thoughtvector/customer-support-on-twitter
License(s): CC-BY-NC-SA-4.0
100% 169M/169M [00:01<00:00, 131MB/s]



In [6]:
from pathlib import Path

    # Find twcs.csv wherever Kaggle extracted it
if Path("/content/twcs/twcs.csv").exists():
        raw_csv_path = Path("/content/twcs/twcs.csv")
elif list(Path("/content").glob("**/twcs*.csv")):
        raw_csv_path = list(Path("/content").glob("**/twcs*.csv"))[0]
else:
        raise FileNotFoundError("twcs.csv not found in /content")

print(f"[OK] Located dataset at: {raw_csv_path}")

[OK] Located dataset at: /content/twcs/twcs.csv


### Step 4: Chunked ETL Pipeline (OOM Prevention)
The `twcs.csv` file contains nearly 3 million tweets (~1GB). Loading the entire dataset into pandas on a 12GB RAM Colab environment causes kernel crashes.
We process the file in streaming chunks of `50,000` rows, filtering specifically for:
- Top reputable brands: `@AppleSupport`, `@AmazonHelp`, `@Uber_Support`, `@SpotifyCares`, `@Delta`, `@AmericanAir`, `@NikeSupport`.
- Inbound inquiries from customers and outbound authoritative replies.



In [7]:
import pandas as pd
import re
from tqdm.auto import tqdm

TARGET_BRANDS = {
    "AppleSupport", "AmazonHelp", "Uber_Support",
    "SpotifyCares", "Delta", "AmericanAir", "NikeSupport"
}

def clean_tweet_text(text: str) -> str:
    """Cleans raw tweet text by removing URLs and standardizing whitespace."""
    if not isinstance(text, str):
        return ""
    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # Anonymize random numeric customer IDs (e.g. @115858 -> @customer)
    text = re.sub(r'@\d+', '@customer', text)
    # Standardize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("[*] Starting chunked scan to build tweet index...")
CHUNK_SIZE = 50_000

# Store tweet mappings: tweet_id -> {text, author, inbound, response_to, brand}
inbound_tweets = {}
brand_responses = []

for chunk in tqdm(pd.read_csv(raw_csv_path, chunksize=CHUNK_SIZE, low_memory=False)):
    # Filter for target brands or responses to/from them
    for _, row in chunk.iterrows():
        t_id = row['tweet_id']
        author = str(row['author_id'])
        inbound = bool(row['inbound'])
        text = str(row['text'])
        response_to = row['in_response_to_tweet_id']

        if inbound:
            # Store customer inquiries that might be answered
            inbound_tweets[t_id] = {
                "text": clean_tweet_text(text),
                "author": author
            }
        else:
            # Check if author is one of our target brands
            if author in TARGET_BRANDS and pd.notna(response_to):
                brand_responses.append({
                    "response_id": t_id,
                    "in_response_to_tweet_id": int(response_to),
                    "brand": author,
                    "response_text": clean_tweet_text(text)
                })

print(f"[OK] Inbound tweets indexed: {len(inbound_tweets):,}")
print(f"[OK] Brand responses collected: {len(brand_responses):,}")



[*] Starting chunked scan to build tweet index...


0it [00:00, ?it/s]

[OK] Inbound tweets indexed: 1,537,843
[OK] Brand responses collected: 457,719


### Step 5: Pairing Inquiries with Responses & JSONL Generation
We match inbound questions with brand answers to build high-quality `(instruction, response)` conversational pairs.



In [8]:
import json
import random

qa_pairs = []

for resp in tqdm(brand_responses, desc="Matching QA pairs"):
    parent_id = resp["in_response_to_tweet_id"]
    if parent_id in inbound_tweets:
        inquiry = inbound_tweets[parent_id]["text"]
        answer = resp["response_text"]
        brand = resp["brand"]

        # Quality filters: minimum length, non-empty, avoid pure redirect bots
        if len(inquiry) >= 20 and len(answer) >= 25:
            if not answer.lower().startswith("please dm us") or len(answer) > 60:
                qa_pairs.append({
                    "instruction": inquiry,
                    "response": answer,
                    "brand": brand,
                    "metadata": {
                        "inquiry_id": parent_id,
                        "response_id": resp["response_id"]
                    }
                })

print(f"[OK] Total high-quality matched pairs: {len(qa_pairs):,}")

# Shuffle and sample 50,000 pairs for balanced training and indexing
random.seed(42)
random.shuffle(qa_pairs)
selected_pairs = qa_pairs[:50_000]

output_jsonl_path = f"{DRIVE_BASE_DIR}/processed/cleaned_customer_support_sample.jsonl"
with open(output_jsonl_path, "w", encoding="utf-8") as f:
    for pair in selected_pairs:
        f.write(json.dumps(pair, ensure_ascii=False) + "\n")

print(f"[OK] Successfully exported {len(selected_pairs):,} pairs to:")
print(f"     {output_jsonl_path}")



Matching QA pairs:   0%|          | 0/457719 [00:00<?, ?it/s]

[OK] Total high-quality matched pairs: 446,642
[OK] Successfully exported 50,000 pairs to:
     /content/drive/MyDrive/chatbot_data/processed/cleaned_customer_support_sample.jsonl


### Step 6: Dataset Summary & Sanity Check
Inspect sample entries and brand distributions.



In [9]:
from collections import Counter

brand_counts = Counter(p["brand"] for p in selected_pairs)
print("Brand distribution in final 50,000 sample:")
for brand, count in brand_counts.most_common():
    print(f"  - {brand:18s}: {count:,} samples ({count/len(selected_pairs)*100:.1f}%)")

print("\nSample Entry Preview:")
print(json.dumps(selected_pairs[0], indent=2))



Brand distribution in final 50,000 sample:
  - AmazonHelp        : 18,333 samples (36.7%)
  - AppleSupport      : 11,829 samples (23.7%)
  - Uber_Support      : 6,218 samples (12.4%)
  - SpotifyCares      : 4,777 samples (9.6%)
  - Delta             : 4,468 samples (8.9%)
  - AmericanAir       : 4,015 samples (8.0%)
  - NikeSupport       : 360 samples (0.7%)

Sample Entry Preview:
{
  "instruction": "@AmazonHelp I will be filing a complaint with the BBB if this issue doesn\u2019t not get resolved today",
  "response": "@customer Once an order is canceled, we don't have the option to send it out. You'll have to place the order again. ^VS",
  "brand": "AmazonHelp",
  "metadata": {
    "inquiry_id": 2343931,
    "response_id": 2343933
  }
}
